# backtesting.py: how many of a backtest's fills were real

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarptandoven/queuetruth/blob/main/examples/notebooks/11_backtesting_py.ipynb)

backtesting.py fills a limit order in full when a bar reaches its price. For backtesting.py the one line is
`queuetruth check` on the run's trades, in Python `qt.check(trades)`: each entry and exit is checked as a limit
order resting at its fill price, against the atlas touch tables or, with order-level data, in the exact queue:

```python
stats._trades.to_csv("trades.csv")   # then: queuetruth check trades.csv --bar 5m
```

Data: queuetruth's **synthetic** dataset and the repository's **synthetic** atlas (invented numbers). The numbers
only show the workflow; they say nothing about a real venue.

In [1]:
# Setup: on Colab (or anywhere queuetruth is missing) this installs it from PyPI; elsewhere it does nothing.
import importlib
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("queuetruth") is None:
    try:
        from IPython import get_ipython

        shell = get_ipython()
    except ImportError:
        shell = None
    if shell is not None:  # %pip lets Colab offer a restart when an already-imported package was upgraded
        shell.run_line_magic("pip", "install -q queuetruth pandas matplotlib")
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "queuetruth", "pandas", "matplotlib"])
    importlib.invalidate_caches()
if importlib.util.find_spec("queuetruth") is None or importlib.util.find_spec("queuetruth.frames") is None:
    raise RuntimeError(
        "this notebook needs queuetruth.frames (first released after queuetruth 0.1.1): pip install -U queuetruth"
    )

In [2]:
import importlib.util
from pathlib import Path

import pandas as pd
from IPython.display import display

import queuetruth as qt

pd.set_option("display.width", 140)
WORK = Path("qt-notebooks") / "11_backtesting_py"
WORK.mkdir(parents=True, exist_ok=True)

# queuetruth's synthetic order-level dataset: two futures (QTAU6, QTBU6), two days, invented prices.
DATA = WORK / "synth"
if not (DATA / "manifest.json").exists():
    qt.fixtures_synth(str(DATA))

# The repository's synthetic atlas (invented numbers, no market data): the checkout's copy, else the one on GitHub.
# Leave `source=` out to use the published atlas (it asks before downloading anything).
_local = Path(qt.__file__).resolve().parents[2] / "tests" / "fixtures" / "atlas_synth"
ATLAS = (
    str(_local)
    if (_local / "atlas" / "v1" / "manifest.json").exists()
    else "https://raw.githubusercontent.com/sarptandoven/queuetruth/main/tests/fixtures/atlas_synth/atlas/v1/"
)

# Twelve resting limit orders on the synthetic futures, as a bar backtest reports them: placed at `placed` (UTC),
# and filled 5 s later when `filled` (the touch rule), else cancelled 30 s later.
ORDERS = pd.DataFrame(
    [
        ("O-1", "QTBU6", "sell", 1, 5008.50, "2026-06-16 14:38:42", True),
        ("O-2", "QTAU6", "buy", 2, 4999.75, "2026-06-16 14:38:05", False),
        ("O-3", "QTBU6", "sell", 3, 5010.25, "2026-06-15 17:09:27", True),
        ("O-4", "QTAU6", "buy", 1, 4999.50, "2026-06-16 17:32:15", False),
        ("O-5", "QTAU6", "sell", 2, 5002.75, "2026-06-15 15:39:13", True),
        ("O-6", "QTAU6", "buy", 3, 5004.00, "2026-06-15 19:04:05", False),
        ("O-7", "QTBU6", "buy", 1, 5008.75, "2026-06-15 15:36:19", True),
        ("O-8", "QTBU6", "sell", 2, 5008.50, "2026-06-16 16:58:59", False),
        ("O-9", "QTAU6", "buy", 3, 4999.50, "2026-06-16 14:39:20", True),
        ("O-10", "QTAU6", "buy", 1, 4999.50, "2026-06-16 17:25:05", False),
        ("O-11", "QTBU6", "buy", 2, 5008.25, "2026-06-16 16:12:29", True),
        ("O-12", "QTAU6", "buy", 3, 5001.25, "2026-06-15 15:05:05", False),
    ],
    columns=["order", "symbol", "side", "qty", "price", "placed", "filled"],
)
ORDERS["placed"] = pd.to_datetime(ORDERS["placed"], utc=True)
ORDERS["done"] = ORDERS["placed"] + pd.to_timedelta(ORDERS["filled"].map({True: 5, False: 30}), unit="s")


def have(*modules, hint=None):
    """True when every module is importable; else prints which are missing and how to get them."""
    missing = [m for m in modules if importlib.util.find_spec(m) is None]
    if missing:
        print(f"skipped: {', '.join(missing)} not installed here ({hint or 'pip install ' + ' '.join(missing)})")
    return not missing


print("queuetruth", qt.__version__, "| SYNTHETIC data and a SYNTHETIC atlas: the numbers show the workflow only")

queuetruth 0.3.0 | SYNTHETIC data and a SYNTHETIC atlas: the numbers show the workflow only


## 1. The trades

With backtesting.py installed, a small strategy on synthetic 5-minute bars writes its trades the usual way. The
check below uses trades of the synthetic future `QTAU6` in the same layout, so it can also replay them in the exact
queue; use your own trades.

In [3]:
if have("backtesting"):
    import numpy as np
    from backtesting import Backtest, Strategy

    idx = pd.date_range("2026-06-15 14:00", periods=288, freq="5min")
    close = 5000 + np.cumsum(np.random.default_rng(0).normal(0, 0.5, len(idx)))
    ohlc = pd.DataFrame({"Open": close, "High": close + 0.5, "Low": close - 0.5, "Close": close}, index=idx)

    class EveryHour(Strategy):
        def init(self):
            pass

        def next(self):
            if len(self.data) % 12 == 0:
                self.position.close()
                self.buy(size=1, limit=self.data.Close[-1] - 0.25)

    stats = Backtest(ohlc, EveryHour, cash=1_000_000, commission=0.0002, finalize_trades=True).run()
    stats._trades.to_csv(WORK / "run_trades.csv")
    print(len(stats._trades), "trades")

trades = pd.DataFrame(
    [
        (-1, 0, 1, 5001.5, 5002.5, -1.02, 0.02, "2026-06-15 15:06:40", "2026-06-15 15:39:26"),
        (-3, 2, 3, 5002.75, 5004.5, -5.27, 0.02, "2026-06-15 15:32:12", "2026-06-15 17:59:24"),
        (-1, 4, 5, 5003.25, 5003.5, -0.27, 0.02, "2026-06-15 17:13:28", "2026-06-15 18:39:00"),
        (-2, 6, 7, 5005.0, 5004.25, 1.48, 0.02, "2026-06-15 17:37:58", "2026-06-15 18:58:54"),
    ],
    columns=["Size", "EntryBar", "ExitBar", "EntryPrice", "ExitPrice", "PnL", "Commission", "EntryTime", "ExitTime"],
)
trades["ReturnPct"] = trades["PnL"] / (trades["EntryPrice"] * trades["Size"].abs())
trades["Duration"] = pd.to_datetime(trades["ExitTime"]) - pd.to_datetime(trades["EntryTime"])
path = WORK / "trades.csv"
trades.to_csv(path)
display(trades)

skipped: backtesting not installed here (pip install backtesting)


,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,PnL,Commission,EntryTime,ExitTime,ReturnPct,Duration
0,-1,0,1,5001.50,5002.50,-1.02,0.02,2026-06-15 15:06:40,2026-06-15 15:39:26,-0.000204,0 days 00:32:46
1,-3,2,3,5002.75,5004.50,-5.27,0.02,2026-06-15 15:32:12,2026-06-15 17:59:24,-0.000351,0 days 02:27:12
2,-1,4,5,5003.25,5003.50,-0.27,0.02,2026-06-15 17:13:28,2026-06-15 18:39:00,-0.000054,0 days 01:25:32
3,-2,6,7,5005.00,5004.25,1.48,0.02,2026-06-15 17:37:58,2026-06-15 18:58:54,0.000148,0 days 01:20:56


## 2. The one line

The trades file does not name its instrument, so `options={"instrument": ...}` names it. Without order-level data,
the atlas estimate (say the bar length; `tier` for symbols the atlas does not hold):

In [4]:
estimate = qt.check(
    path, bar="5m", tier="ALL", atlas=ATLAS, yes=True, options={"instrument": "QTAU6"}, out=WORK / "estimate"
)
display(estimate)

tier,tier_name,side,bar,orders,backtest_filled,p_real_given_touch,p_real_given_touch_ci_lo,p_real_given_touch_ci_hi,expected_real_fills,pnl_overstated_ticks_per_order,pnl_overstated_ticks_per_order_ci_lo,pnl_overstated_ticks_per_order_ci_hi,atlas_orders,atlas_dates,atlas_symbols
ALL,all symbols,buy,5m,4,4,0.55,0.5,0.6,2.2,0.4,0.3,0.5,4000,3,40
ALL,all symbols,sell,5m,4,4,0.57,0.52,0.62,2.28,0.4,0.3,0.5,4000,3,40


With order-level data of the same day, each entry and exit is replayed in the exact queue at its price and time:

In [5]:
res = qt.check(path, data=DATA, horizon="1s", window="10s", options={"instrument": "QTAU6"}, out=WORK / "check")
display(res)

order,symbol,side,price,qty,entry,entry_local,backtest_filled,backtest_fill_after_s,real_filled,real_fill_after_s,queue_ahead,outcome,pnl_gap
sim:backtesting_py:backtest:QTAU6-f0-t0-entry,QTAU6,sell,5001.5,1,2026-06-15T15:06:40.000+00:00,2026-06-15 15:06:40.000 UTC,True,0,False,,75,phantom,6.24
sim:backtesting_py:backtest:QTAU6-f0-t0-exit,QTAU6,buy,5002.5,1,2026-06-15T15:39:26.000+00:00,2026-06-15 15:39:26.000 UTC,True,0,True,2.736,0,both,6.24571
sim:backtesting_py:backtest:QTAU6-f0-t1-entry,QTAU6,sell,5002.75,3,2026-06-15T15:32:12.000+00:00,2026-06-15 15:32:12.000 UTC,True,0,False,,3,phantom,18.74
sim:backtesting_py:backtest:QTAU6-f0-t1-exit,QTAU6,buy,5004.5,3,2026-06-15T17:59:24.000+00:00,2026-06-15 17:59:24.000 UTC,True,0,False,,0,phantom,-0.0100017
sim:backtesting_py:backtest:QTAU6-f0-t2-entry,QTAU6,sell,5003.25,1,2026-06-15T17:13:28.000+00:00,2026-06-15 17:13:28.000 UTC,True,0,False,,0,phantom,6.24
sim:backtesting_py:backtest:QTAU6-f0-t2-exit,QTAU6,buy,5003.5,1,2026-06-15T18:39:00.000+00:00,2026-06-15 18:39:00.000 UTC,True,0,True,3.407,10,both,12.4957
sim:backtesting_py:backtest:QTAU6-f0-t3-entry,QTAU6,sell,5005,2,2026-06-15T17:37:58.000+00:00,2026-06-15 17:37:58.000 UTC,True,0,False,,3,phantom,12.49
sim:backtesting_py:backtest:QTAU6-f0-t3-exit,QTAU6,buy,5004.25,2,2026-06-15T18:58:54.000+00:00,2026-06-15 18:58:54.000 UTC,True,0,True,8.018,21,both,12.5014


In [6]:
display(res.table()[["symbol", "side", "price", "entry_local", "backtest_filled", "real_filled", "outcome"]])

,symbol,side,price,entry_local,backtest_filled,real_filled,outcome
0,QTAU6,sell,5001.5,2026-06-15 15:06:40.000 UTC,True,False,phantom
1,QTAU6,buy,5002.5,2026-06-15 15:39:26.000 UTC,True,True,both
2,QTAU6,sell,5002.75,2026-06-15 15:32:12.000 UTC,True,False,phantom
3,QTAU6,buy,5004.5,2026-06-15 17:59:24.000 UTC,True,False,phantom
4,QTAU6,sell,5003.25,2026-06-15 17:13:28.000 UTC,True,False,phantom
5,QTAU6,buy,5003.5,2026-06-15 18:39:00.000 UTC,True,True,both
6,QTAU6,sell,5005,2026-06-15 17:37:58.000 UTC,True,False,phantom
7,QTAU6,buy,5004.25,2026-06-15 18:58:54.000 UTC,True,True,both
